# 07 — Distribution Shift in Time Series

## The core problem

Imagine you train a stock forecasting model on data from January to June.
In July, the market regime changes — prices jump to a new level, volatility
spikes, or a trend reverses.

Your model was trained on one **distribution** and now faces a different one.
This is called **distribution shift** (or **covariate shift** / **concept drift**).

## Types of shift

| Type | What changes | Example |
|------|-------------|----------|
| **Mean shift** | The average level changes | Stock gaps up from $100 to $150 |
| **Variance shift** | The spread/volatility changes | Calm market → volatile market |
| **Trend shift** | Direction reverses | Bull → Bear market |

## Why normalization doesn't fully solve this

**Global normalization** (normalizing with statistics from the entire training set)
fails when test data has a different mean or variance. The model's "mental model"
of normal values doesn't transfer to the new regime.

→ **RevIN** solves this by normalizing *each input window independently*
using only its own statistics — no global assumptions needed.

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.plotting import plot_before_after_distribution, plot_line, plot_rolling_stats

In [ ]:
df_orig    = pd.read_csv('../data/stock_sample.csv')
df_shifted = pd.read_csv('../data/shifted_stock_sample.csv')

orig_close    = df_orig['stock_price'].values    if 'stock_price' in df_orig.columns else df_orig['close'].values
shifted_close = df_shifted['close'].values
orig_close    = df_orig['close'].values

## Step 1 — Mean shift

The shifted dataset has a much higher starting price (different mean level).

In [ ]:
print(f'Original   — mean: {orig_close.mean():.2f},  std: {orig_close.std():.2f}')
print(f'Shifted    — mean: {shifted_close.mean():.2f}, std: {shifted_close.std():.2f}')
print(f'Mean shift : +{shifted_close.mean() - orig_close.mean():.2f} ({100*(shifted_close.mean()-orig_close.mean())/orig_close.mean():.0f}%)')
print(f'Std ratio  :  {shifted_close.std() / orig_close.std():.2f}x')

In [ ]:
plot_before_after_distribution(
    orig_close, shifted_close,
    label_before='Original distribution',
    label_after='Shifted distribution',
    title='Distribution Shift — Original vs Shifted Stock'
)

## Step 2 — Simulate shift: concat the two series

In [ ]:
combined = np.concatenate([orig_close, shifted_close])
t        = np.arange(len(combined))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[:len(orig_close)],    orig_close,    color='#4C72B0', label='Original regime')
ax.plot(t[len(orig_close):],    shifted_close, color='#DD8452', label='Shifted regime')
ax.axvline(len(orig_close), color='red', linestyle='--', linewidth=1.5, label='Regime change')
ax.set_title('Simulated Distribution Shift over Time', fontsize=13)
ax.set_xlabel('Time step')
ax.set_ylabel('Price ($)')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## Step 3 — Why global normalization fails here

In [ ]:
# Global normalization trained on ORIGINAL data only
global_mean = orig_close.mean()
global_std  = orig_close.std()

orig_norm    = (orig_close    - global_mean) / global_std
shifted_norm = (shifted_close - global_mean) / global_std   # wrong stats!

print(f'Original (global norm)  — mean: {orig_norm.mean():.2f},   std: {orig_norm.std():.2f}')
print(f'Shifted  (global norm)  — mean: {shifted_norm.mean():.2f}, std: {shifted_norm.std():.2f}')
print()
print('The shifted series is NOT centred at 0 — the model sees an out-of-distribution signal.')

## Step 4 — Instance normalization handles it

In [ ]:
# Normalize each window independently (per-instance)
WINDOW = 20
windows = [shifted_close[i:i+WINDOW] for i in range(0, len(shifted_close) - WINDOW, WINDOW)]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, window in zip(axes, windows[:3]):
    normed = (window - window.mean()) / (window.std() + 1e-5)
    ax.plot(normed, 'o-', color='#55A868')
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    ax.set_title(f'Instance norm\nμ={normed.mean():.2f}, σ={normed.std():.2f}', fontsize=11)
    ax.set_xlabel('Step')
    ax.spines[['top','right']].set_visible(False)

axes[0].set_ylabel('Normalized value')
fig.suptitle('Each window independently centred at 0 — regardless of regime', fontsize=13)
plt.tight_layout()
plt.show()

## Step 5 — Rolling statistics reveal non-stationarity

In [ ]:
plot_rolling_stats(combined, window=15, title='Rolling Stats — Regime Change is Visible', ylabel='Price ($)')

The rolling mean jumps at the regime boundary, and the rolling std increases
in the second half — clear evidence of non-stationarity.

---
## Exercises

1. Simulate a **variance shift only** (same mean, higher std). How does the histogram look?
2. What would happen to model performance if you applied global normalization
   to the shifted data? Would RMSE go up or down?
3. Look at the rolling std plot. At what time step does the shift become obvious?

---
**Next →** `08_revin_from_scratch.ipynb` — the solution.